# SMS Spam Detection — Machine Learning Experiment

Notebook chính của dự án. Thực hiện lần lượt: **đọc dữ liệu → làm sạch → chia Train/Test → TF-IDF → huấn luyện → đánh giá → xuất kết quả → thử dự đoán**.

- Bộ dữ liệu huấn luyện: **UCI SMS Spam Collection (English)**.
- `app.py` chỉ tải Pipeline đã lưu; không cần chạy Notebook mỗi lần mở giao diện.
- Không sử dụng tập Test để huấn luyện hoặc chọn siêu tham số.

In [1]:
import csv
from pathlib import Path
import joblib
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

# Notebook đặt tại thư mục gốc dự án.
BASE_DIR = Path.cwd().resolve()
DATA_PATH = BASE_DIR / "data" / "sms+spam+collection" / "SMSSpamCollection"
MODELS_DIR = BASE_DIR / "models"
OUTPUTS_DIR = BASE_DIR / "outputs"
assert DATA_PATH.is_file(), f"Không tìm thấy dataset: {DATA_PATH}. Hãy mở Notebook từ thư mục gốc dự án."
MODELS_DIR.mkdir(parents=True, exist_ok=True)
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
print("Project:", BASE_DIR)

Project: C:\ML\BTL\sms-spam-detection-converted


## 1. Đọc và làm sạch dữ liệu

Loại bỏ các hàng trùng hoàn toàn, không sửa nội dung SMS và nhãn gốc.

In [2]:
def load_dataset():
    """
    Đọc dataset SMS Spam Collection.

    Returns:
        DataFrame gồm 2 cột: label va message.
    """
    df = pd.read_csv(
        DATA_PATH,
        sep="\t",
        header=None,
        names=["label", "message"],
        encoding="utf-8",
        quoting=csv.QUOTE_NONE,  # Dấu ngoặc kép là một phần nội dung SMS.
        keep_default_na=False
    )

    return df


def clean_dataset(df):
    """
    Loai bo cac dong trung lap hoan toan.
    Khong thay doi DataFrame goc.

    Args:
        df: DataFrame chua du lieu SMS.

    Returns:
        DataFrame da loai bo trung lap.
    """
    df_clean = df.drop_duplicates().copy()

    # Danh lai chi so dong tu 0
    df_clean = df_clean.reset_index(drop=True)

    return df_clean


def split_dataset(df, test_size=0.2, random_state=42):
    """
    Chia du lieu thanh tap train va test.

    Args:
        df: DataFrame da lam sach.
        test_size: Ty le du lieu danh cho tap test.
        random_state: Co dinh ket qua chia du lieu.

    Returns:
        X_train, X_test, y_train, y_test
    """

    # X: Noi dung tin nhan
    X = df["message"]

    # y: Nhan phan loai (ham hoac spam)
    y = df["label"]

    # Chia du lieu theo ty le 80/20
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=test_size,
        random_state=random_state,
        stratify=y
    )

    return X_train, X_test, y_train, y_test

In [3]:
df = load_dataset()
df_clean = clean_dataset(df)
print(f"Ban đầu: {len(df)} | Sau loại trùng: {len(df_clean)} | Đã loại: {len(df)-len(df_clean)}")
print("Phân bố nhãn:", df_clean['label'].value_counts())
display(df_clean.head())

Ban đầu: 5574 | Sau loại trùng: 5171 | Đã loại: 403
Phân bố nhãn: label
ham     4518
spam     653
Name: count, dtype: int64


,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


## 2. Chia dữ liệu Train/Test

Chia 80/20, phân tầng theo nhãn (`stratify=y`), cố định `random_state=42`.

In [4]:
X_train, X_test, y_train, y_test = split_dataset(df_clean)
print("Train:", len(X_train), "| Test:", len(X_test))
print("Train labels:", y_train.value_counts())
print("Test labels:", y_test.value_counts())

Train: 4136 | Test: 1035
Train labels: label
ham     3614
spam     522
Name: count, dtype: int64
Test labels: label
ham     904
spam    131
Name: count, dtype: int64


## 3. TF-IDF và cấu hình hai Pipeline

TF-IDF sử dụng unigram/bigram, `min_df=2`, `max_df=0.95`, `sublinear_tf=True`. Mỗi Pipeline tự học TF-IDF **chỉ trên Train** khi gọi `fit`, tránh rò rỉ dữ liệu Test.

In [5]:
def create_tfidf():
    """
    Khoi tao bo trich xuat dac trung TF-IDF.

    Returns:
        TfidfVectorizer chua duoc huan luyen.
    """
    vectorizer = TfidfVectorizer(
        lowercase=True,
        ngram_range=(1, 2),
        min_df=2,
        max_df=0.95,
        sublinear_tf=True
    )

    return vectorizer


def create_pipelines():
    """
    Khoi tao hai Machine Learning Pipeline:
    1. TF-IDF + Multinomial Naive Bayes
    2. TF-IDF + Logistic Regression

    Returns:
        Dictionary chua hai Pipeline chua huan luyen.
    """
    pipelines = {
        "Multinomial Naive Bayes": Pipeline([
            ("tfidf", create_tfidf()),
            ("classifier", MultinomialNB())
        ]),

        "Logistic Regression": Pipeline([
            ("tfidf", create_tfidf()),
            ("classifier", LogisticRegression(
                max_iter=1000,
                random_state=42
            ))
        ])
    }

    return pipelines


def train_models(pipelines, X_train, y_train):
    """
    Huan luyen cac Machine Learning Pipeline.

    Args:
        pipelines: Dictionary chua hai Pipeline.
        X_train: Noi dung SMS cua tap Train.
        y_train: Nhan Ham/Spam cua tap Train.

    Returns:
        Dictionary chua cac Pipeline da huan luyen.
    """
    trained_models = {}

    for name, pipeline in pipelines.items():
        print(f"\nDang huan luyen: {name}")

        # Pipeline tu dong chay TF-IDF roi huan luyen bo phan loai
        pipeline.fit(X_train, y_train)

        trained_models[name] = pipeline

        print(f"Huan luyen {name} thanh cong!")

    return trained_models


def save_models(trained_models):
    """
    Luu cac Pipeline da huan luyen vao thu muc models/.

    Args:
        trained_models: Dictionary chua hai Pipeline da huan luyen.
    """

    # Xac dinh duong dan thu muc models
    base_dir = BASE_DIR
    models_dir = base_dir / "models"

    # Tao thu muc neu chua ton tai
    models_dir.mkdir(parents=True, exist_ok=True)

    # Dat ten file cho tung mo hinh
    file_names = {
        "Multinomial Naive Bayes": "naive_bayes_pipeline.joblib",
        "Logistic Regression": "logistic_regression_pipeline.joblib"
    }

    # Luu tung Pipeline
    for name, model in trained_models.items():
        file_path = models_dir / file_names[name]

        joblib.dump(model, file_path)

        print(f"Da luu mo hinh {name}: {file_path}")

In [6]:
# Khảo sát kích thước đặc trưng trên Train; KHÔNG fit trên Test.
preview_vectorizer = create_tfidf()
X_train_tfidf = preview_vectorizer.fit_transform(X_train)
X_test_tfidf = preview_vectorizer.transform(X_test)
print("Train TF-IDF:", X_train_tfidf.shape)
print("Test TF-IDF:", X_test_tfidf.shape)
print("Vocabulary size:", len(preview_vectorizer.vocabulary_))
pipelines = create_pipelines()
for name, pipeline in pipelines.items():
    print(name, pipeline)

Train TF-IDF: (4136, 10164)
Test TF-IDF: (1035, 10164)
Vocabulary size: 10164
Multinomial Naive Bayes Pipeline(steps=[('tfidf',
                 TfidfVectorizer(max_df=0.95, min_df=2, ngram_range=(1, 2),
                                 sublinear_tf=True)),
                ('classifier', MultinomialNB())])
Logistic Regression Pipeline(steps=[('tfidf',
                 TfidfVectorizer(max_df=0.95, min_df=2, ngram_range=(1, 2),
                                 sublinear_tf=True)),
                ('classifier',
                 LogisticRegression(max_iter=1000, random_state=42))])


## 4. Huấn luyện

Huấn luyện Multinomial Naive Bayes và Logistic Regression theo cấu hình gốc.

In [7]:
trained_models = train_models(pipelines, X_train, y_train)
print("Số mô hình:", len(trained_models))


Dang huan luyen: Multinomial Naive Bayes
Huan luyen Multinomial Naive Bayes thanh cong!

Dang huan luyen: Logistic Regression
Huan luyen Logistic Regression thanh cong!
Số mô hình: 2


## 5. Đánh giá và trực quan hóa

Các chỉ số Precision, Recall và F1 sử dụng **Spam là lớp dương**; ma trận nhầm lẫn theo thứ tự `[ham, spam]`.

In [8]:
def evaluate_model(model, X_test, y_test):
    """
    Danh gia mo hinh tren tap Test.

    Args:
        model: Pipeline da duoc huan luyen.
        X_test: Noi dung SMS cua tap Test.
        y_test: Nhan thuc te cua tap Test.

    Returns:
        Dictionary chua cac chi so danh gia.
    """

    # Du doan nhan Ham/Spam tren tap Test
    spam_idx = list(model.classes_).index("spam")
    spam_probs = model.predict_proba(X_test)[:, spam_idx]
    y_pred = ["spam" if prob >= 0.5 else "ham" for prob in spam_probs]

    # Tinh cac chi so, xem Spam la lop duong tinh
    metrics = {
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(
            y_test, y_pred, pos_label="spam", zero_division=0
        ),
        "recall": recall_score(
            y_test, y_pred, pos_label="spam", zero_division=0
        ),
        "f1_score": f1_score(
            y_test, y_pred, pos_label="spam", zero_division=0
        ),
        "confusion_matrix": confusion_matrix(
            y_test, y_pred, labels=["ham", "spam"]
        )
    }

    return metrics


def plot_confusion_matrix(cm, model_name, output_path):
    """Vẽ số tin và tỷ lệ theo nhãn thực tế ở ngưỡng 0,50."""
    fig, ax = plt.subplots(figsize=(6.3, 4.3), layout="constrained")
    ax.set_axis_off()
    ax.set_title(f"Confusion matrix | {model_name}", loc="left", fontsize=14, weight="bold", pad=12)

    labels = [["TN", "FP"], ["FN", "TP"]]
    rows = []
    for actual, row in zip(["Ham", "Spam"], cm):
        total = row.sum()
        cells = [
            f"{labels[len(rows)][column]}  ·  {count:,} SMS\n{count / total:.1%} of actual {actual}"
            if total else f"{labels[len(rows)][column]}  ·  {count:,} SMS\nNo actual {actual}"
            for column, count in enumerate(row)
        ]
        rows.append([actual, *cells])

    table = ax.table(
        cellText=rows,
        colLabels=["Actual label", "Predicted Ham", "Predicted Spam"],
        colWidths=[0.24, 0.38, 0.38],
        cellLoc="center",
        bbox=[0, 0.20, 1, 0.71],
    )
    table.auto_set_font_size(False)
    table.set_fontsize(11)
    for (row, column), cell in table.get_celld().items():
        cell.set_edgecolor("white")
        cell.set_linewidth(2)
        if row == 0:
            cell.set_facecolor("#24364B")
            cell.set_text_props(color="white", weight="bold")
        elif column == 0:
            cell.set_facecolor("#EFF3F7")
            cell.set_text_props(color="#24364B", weight="bold")
        elif row == column:
            cell.set_facecolor("#DDF3EA")
            cell.set_text_props(color="#14594B")
        else:
            cell.set_facecolor("#FBE9DB")
            cell.set_text_props(color="#793C19")

    ax.text(0, 0.10, "Green = correct prediction  |  Orange = classification error",
            transform=ax.transAxes, fontsize=10, color="#24364B")
    ax.text(0, 0.03, f"Spam threshold 0.50  ·  Test set: {cm.sum():,} SMS",
            transform=ax.transAxes, fontsize=10, color="#475569")
    fig.savefig(output_path, dpi=300)
    plt.close(fig)
    print(f"Da luu bieu do: {output_path}")


def plot_model_comparison(metrics_df, output_path):
    """So sánh chỉ số giữa hai mô hình bằng các cặp thanh ngang."""
    columns = ["accuracy", "precision", "recall", "f1_score"]
    labels = ["Accuracy", "Precision (Spam)", "Recall (Spam)", "F1-score (Spam)"]
    positions = list(range(len(columns)))
    fig, ax = plt.subplots(figsize=(9.4, 5.4), layout="constrained")
    ax.set_title("Model performance comparison", loc="left", fontsize=16, weight="bold", pad=36)

    for offset, color, (_, model) in zip([-0.18, 0.18], ["#176B87", "#A74726"], metrics_df.iterrows()):
        bars = ax.barh(
            [position + offset for position in positions],
            model[columns].to_numpy(dtype=float) * 100,
            height=0.32,
            color=color,
            label=model["model"],
        )
        ax.bar_label(bars, fmt="%.2f%%", padding=5, fontsize=10, color="#24364B")

    ax.set_yticks(positions, labels)
    ax.invert_yaxis()
    ax.set_xlim(0, 112)
    ax.set_xticks(range(0, 101, 20), [f"{value}%" for value in range(0, 101, 20)])
    ax.set_xlabel("Test-set score  ·  Spam threshold 0.50", labelpad=10)
    ax.set_axisbelow(True)
    ax.grid(axis="x", color="#DFE6ED")
    ax.tick_params(length=0, pad=7)
    ax.spines[["top", "right", "left", "bottom"]].set_visible(False)
    ax.legend(loc="lower left", bbox_to_anchor=(0, 1.01), frameon=False, ncol=2)
    fig.savefig(output_path, dpi=300)
    plt.close(fig)
    print(f"Da luu bieu do so sanh: {output_path}")


In [9]:
metrics_records = []
for name, model in trained_models.items():
    results = evaluate_model(model, X_test, y_test)
    metrics_records.append({
        "model": name,
        "accuracy": results["accuracy"],
        "precision": results["precision"],
        "recall": results["recall"],
        "f1_score": results["f1_score"],
    })
    print(f"\n{name}: Accuracy={results['accuracy']:.4f}, Precision={results['precision']:.4f}, Recall={results['recall']:.4f}, F1={results['f1_score']:.4f}")
    print("Confusion Matrix [ham, spam]:\n", results["confusion_matrix"])
    image_name = ("confusion_matrix_naive_bayes.png" if name == "Multinomial Naive Bayes"
                  else "confusion_matrix_logistic_regression.png")
    plot_confusion_matrix(results["confusion_matrix"], name, OUTPUTS_DIR / image_name)

metrics_df = pd.DataFrame(metrics_records)
metrics_df.to_csv(OUTPUTS_DIR / "metrics.csv", index=False)
plot_model_comparison(metrics_df, OUTPUTS_DIR / "model_comparison.png")
display(metrics_df)


Multinomial Naive Bayes: Accuracy=0.9633, Precision=1.0000, Recall=0.7099, F1=0.8304
Confusion Matrix [ham, spam]:
 [[904   0]
 [ 38  93]]
Da luu bieu do: C:\ML\BTL\sms-spam-detection-converted\outputs\confusion_matrix_naive_bayes.png

Logistic Regression: Accuracy=0.9633, Precision=1.0000, Recall=0.7099, F1=0.8304
Confusion Matrix [ham, spam]:
 [[904   0]
 [ 38  93]]
Da luu bieu do: C:\ML\BTL\sms-spam-detection-converted\outputs\confusion_matrix_logistic_regression.png
Da luu bieu do so sanh: C:\ML\BTL\sms-spam-detection-converted\outputs\model_comparison.png


,model,accuracy,precision,recall,f1_score
0,Multinomial Naive Bayes,0.963285,1.0,0.709924,0.830357
1,Logistic Regression,0.963285,1.0,0.709924,0.830357


**Nhận xét:** Cần xem đồng thời Precision và Recall của lớp Spam. Precision cao không đồng nghĩa phát hiện được tất cả Spam; Recall thấp tương ứng với nhiều tin Spam bị bỏ sót. Kết quả thực tế nằm trong bảng trên.

### 5.1. Error Analysis – Phân tích các trường hợp dự đoán sai

Phân tích các tin nhắn bị phân loại nhầm trên tập Test,
đặc biệt là Spam bị dự đoán thành Ham (False Negative).

In [10]:
error_results = {}

for name, model in trained_models.items():

    # Du doan tren tap Test
    spam_idx = list(model.classes_).index("spam")
    spam_probs = model.predict_proba(X_test)[:, spam_idx]
    y_pred = ["spam" if prob >= 0.5 else "ham" for prob in spam_probs]

    # Tao bang so sanh nhan thuc te va du doan
    analysis_df = pd.DataFrame({
        "message (Tin nhắn)": X_test.to_numpy(),
        "actual (Thực tế)": y_test.to_numpy(),
        "predicted (Dự đoán)": y_pred
    })

    # Loc cac tin nhan du doan sai
    errors = analysis_df[
        analysis_df["actual (Thực tế)"] != analysis_df["predicted (Dự đoán)"]
    ].copy()

    # Spam bi du doan nham thanh Ham
    false_negatives = errors[
        (errors["actual (Thực tế)"] == "spam") &
        (errors["predicted (Dự đoán)"] == "ham")
    ]

    # Ham bi du doan nham thanh Spam
    false_positives = errors[
        (errors["actual (Thực tế)"] == "ham") &
        (errors["predicted (Dự đoán)"] == "spam")
    ]

    error_results[name] = errors

    print(f"\nModel: {name}")
    print(f"Total errors (Tổng số lỗi): {len(errors)}")
    print(f"False Negatives (Âm tính giả): {len(false_negatives)}")
    print(f"False Positives (Dương tính giả): {len(false_positives)}")

    print("\nSample False Negatives (Âm tính giả):")
    display(false_negatives.head(5))



Model: Multinomial Naive Bayes
Total errors (Tổng số lỗi): 38
False Negatives (Âm tính giả): 38
False Positives (Dương tính giả): 0

Sample False Negatives (Âm tính giả):


,message (Tin nhắn),actual (Thực tế),predicted (Dự đoán)
121,"FreeMsg Hey U, i just got 1 of these video/pic...",spam,ham
141,Check Out Choose Your Babe Videos @ sms.shsex....,spam,ham
168,You will recieve your tone within the next 24h...,spam,ham
189,Burger King - Wanna play footy at a top stadiu...,spam,ham
223,Xmas & New Years Eve tickets are now on sale f...,spam,ham



Model: Logistic Regression
Total errors (Tổng số lỗi): 38
False Negatives (Âm tính giả): 38
False Positives (Dương tính giả): 0

Sample False Negatives (Âm tính giả):


,message (Tin nhắn),actual (Thực tế),predicted (Dự đoán)
121,"FreeMsg Hey U, i just got 1 of these video/pic...",spam,ham
141,Check Out Choose Your Babe Videos @ sms.shsex....,spam,ham
168,You will recieve your tone within the next 24h...,spam,ham
189,Burger King - Wanna play footy at a top stadiu...,spam,ham
205,Sunshine Quiz Wkly Q! Win a top Sony DVD playe...,spam,ham


### 5.2. Compare prediction errors between models - So sánh lỗi dự đoán giữa 2 mô hình

In [11]:
nb_errors = error_results["Multinomial Naive Bayes"]
lr_errors = error_results["Logistic Regression"]

# Lay chi so cac tin nhan du doan sai
nb_error_indices = set(nb_errors.index)
lr_error_indices = set(lr_errors.index)

# Cac truong hop ca hai mo hinh deu du doan sai
common_errors = nb_error_indices & lr_error_indices

# Cac truong hop chi mot mo hinh du doan sai
nb_only = nb_error_indices - lr_error_indices
lr_only = lr_error_indices - nb_error_indices

print("Common errors (lỗi chung):", len(common_errors))
print("Naive Bayes only (chỉ Naive Bayes):", len(nb_only))
print("Logistic Regression only (chỉ Logistic Regression):", len(lr_only))

# Hien thi cac tin nhan ca hai mo hinh deu du doan sai
common_errors_df = nb_errors.loc[
    sorted(common_errors)
]

print("\nSample common errors (lỗi chung):")
display(common_errors_df.head(10))


Common errors (lỗi chung): 29
Naive Bayes only (chỉ Naive Bayes): 9
Logistic Regression only (chỉ Logistic Regression): 9

Sample common errors (lỗi chung):


,message (Tin nhắn),actual (Thực tế),predicted (Dự đoán)
121,"FreeMsg Hey U, i just got 1 of these video/pic...",spam,ham
141,Check Out Choose Your Babe Videos @ sms.shsex....,spam,ham
168,You will recieve your tone within the next 24h...,spam,ham
189,Burger King - Wanna play footy at a top stadiu...,spam,ham
223,Xmas & New Years Eve tickets are now on sale f...,spam,ham
231,RECPT 1/3. You have ordered a Ringtone. Your o...,spam,ham
248,ringtoneking 84484,spam,ham
255,Would you like to see my XXX pics they are so ...,spam,ham
263,Dear Voucher Holder 2 claim your 1st class air...,spam,ham
268,Want explicit SEX in 30 secs? Ring 02073162414...,spam,ham


### 5.3. Đọc kết quả phân tích lỗi

Các ô phía trên tính số lỗi, số lỗi chung và số lỗi riêng trên lần chạy hiện tại.
Âm tính giả là Spam bị phân loại thành Ham. Dương tính giả là Ham bị phân loại thành Spam.
Đọc các tin nhắn mẫu để tìm đặc điểm chung của những trường hợp mô hình bỏ sót.

## 6. Lưu Pipeline và thử dự đoán

Lưu **Pipeline gồm TF-IDF + classifier** để `app.py` dự đoán trực tiếp từ văn bản gốc.

In [12]:
save_models(trained_models)

def load_model(model_name="naive_bayes"):
    model_files = {
        "naive_bayes": "naive_bayes_pipeline.joblib",
        "logistic_regression": "logistic_regression_pipeline.joblib",
    }
    if model_name not in model_files:
        raise ValueError(f"Mô hình không hợp lệ: {model_name}")
    return joblib.load(MODELS_DIR / model_files[model_name])

def predict_message(model, message):
    prediction = model.predict([message])[0]
    probabilities = model.predict_proba([message])[0]
    return prediction, dict(zip(model.classes_, probabilities))

test_message = "Congratulations! You have won a free prize. Claim now!"
for model_name in ["naive_bayes", "logistic_regression"]:
    prediction, probabilities = predict_message(load_model(model_name), test_message)
    print(f"{model_name}: {prediction} | Ham={probabilities['ham']:.4f} | Spam={probabilities['spam']:.4f}")

Da luu mo hinh Multinomial Naive Bayes: C:\ML\BTL\sms-spam-detection-converted\models\naive_bayes_pipeline.joblib
Da luu mo hinh Logistic Regression: C:\ML\BTL\sms-spam-detection-converted\models\logistic_regression_pipeline.joblib
naive_bayes: spam | Ham=0.0845 | Spam=0.9155
logistic_regression: spam | Ham=0.1219 | Spam=0.8781


## 7. Threshold Tuning – Điều chỉnh ngưỡng phân loại Spam

Thử nghiệm các ngưỡng phân loại khác nhau nhằm giảm số lượng
tin nhắn Spam bị bỏ sót, đồng thời theo dõi Precision, Recall và F1-score.

Việc lựa chọn ngưỡng sẽ sử dụng tập Validation riêng,
không sử dụng tập Test để tối ưu ngưỡng.

### 7.1. Checking data before creating a validation split - Kiểm tra dữ liệu trước khi chia Validation

In [13]:
print("X_train:", type(X_train), len(X_train))
print("y_train:", type(y_train), len(y_train))

print("X_test:", type(X_test), len(X_test))
print("y_test:", type(y_test), len(y_test))

print("\nTrain label distribution (Phân phối nhãn train):")
print(y_train.value_counts())

print("\nTest label distribution (Phân phối nhãn test):")
print(y_test.value_counts())

X_train: <class 'pandas.core.series.Series'> 4136
y_train: <class 'pandas.core.series.Series'> 4136
X_test: <class 'pandas.core.series.Series'> 1035
y_test: <class 'pandas.core.series.Series'> 1035

Train label distribution (Phân phối nhãn train):
label
ham     3614
spam     522
Name: count, dtype: int64

Test label distribution (Phân phối nhãn test):
label
ham     904
spam    131
Name: count, dtype: int64


### 7.2. Split the training set into a training subset and a validation subset - Chia Train thành tập con train và tập con validation

In [14]:
from sklearn.model_selection import train_test_split

X_train_sub, X_val, y_train_sub, y_val = train_test_split(
    X_train,
    y_train,
    test_size=0.2,
    random_state=42,
    stratify=y_train
)

print("Training subset:", len(X_train_sub))
print("Validation subset:", len(X_val))
print("Test set (unchanged):", len(X_test))

print("\nTraining subset label distribution (Phân phối nhãn train):")
print(y_train_sub.value_counts())

print("\nValidation subset label distribution (Phân phối nhãn tập xác thực):")
print(y_val.value_counts())


Training subset: 3308
Validation subset: 828
Test set (unchanged): 1035

Training subset label distribution (Phân phối nhãn train):
label
ham     2891
spam     417
Name: count, dtype: int64

Validation subset label distribution (Phân phối nhãn tập xác thực):
label
ham     723
spam    105
Name: count, dtype: int64


### 7.3. Training the pipeline on the training subset - Huấn luyện Pipeline thử nghiệm trên Training subset

In [15]:
from sklearn.base import clone

validation_models = {}

for model_name, model in trained_models.items():

    # Tao ban sao Pipeline chua huan luyen
    validation_model = clone(model)

    # Chi huan luyen tren Training subset
    validation_model.fit(X_train_sub, y_train_sub)

    # Luu trong bo nho, khong ghi de file .joblib
    validation_models[model_name] = validation_model

    print(f"Trained validation model: {model_name}")

print("\nTotal validation models:", len(validation_models))

Trained validation model: Multinomial Naive Bayes
Trained validation model: Logistic Regression

Total validation models: 2


### 7.4. Evaluating thresholds on the validation subset - Đánh giá các ngưỡng trên Validation subset

In [16]:
from sklearn.metrics import precision_score, recall_score, f1_score

thresholds = [0.3, 0.4, 0.5]
threshold_results = []

for model_name, model in validation_models.items():

    # Lay xac suat Spam
    spam_idx = list(model.classes_).index("spam")
    spam_probs = model.predict_proba(X_val)[:, spam_idx]

    for threshold in thresholds:

        # Chuyen xac suat thanh nhan du doan
        y_pred = [
            "spam" if prob >= threshold else "ham"
            for prob in spam_probs
        ]

        # Tinh cac chi so
        precision = precision_score(
            y_val, y_pred, pos_label="spam", zero_division=0
        )
        recall = recall_score(
            y_val, y_pred, pos_label="spam", zero_division=0
        )
        f1 = f1_score(
            y_val, y_pred, pos_label="spam", zero_division=0
        )

        threshold_results.append({
            "Model": model_name,
            "Threshold": threshold,
            "Precision": precision,
            "Recall": recall,
            "F1-score": f1
        })

threshold_df = pd.DataFrame(threshold_results)

display(
    threshold_df.round(4)
)

,Model,Threshold,Precision,Recall,F1-score
0,Multinomial Naive Bayes,0.3,1.0000,0.8190,0.9005
1,Multinomial Naive Bayes,0.4,1.0000,0.7619,0.8649
2,Multinomial Naive Bayes,0.5,1.0000,0.7048,0.8268
3,Logistic Regression,0.3,0.9674,0.8476,0.9036
4,Logistic Regression,0.4,0.9770,0.8095,0.8854
5,Logistic Regression,0.5,0.9877,0.7619,0.8602


### 7.5. Evaluating the selected threshold on the test set - Đánh giá ngưỡng đã chọn trên Test set

In [17]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

# Chọn ngưỡng có F1 cao nhất cho từng mô hình trên Validation.
# Khi F1 bằng nhau, ưu tiên ngưỡng cao hơn.
selected_thresholds = (
    threshold_df.sort_values(["F1-score", "Threshold"], ascending=False)
    .drop_duplicates("Model")
    .set_index("Model")["Threshold"]
)
final_results = []

for model_name, model in trained_models.items():

    # Lay xac suat Spam tren Test
    spam_idx = list(model.classes_).index("spam")
    spam_probs = model.predict_proba(X_test)[:, spam_idx]

    selected_threshold = selected_thresholds.loc[model_name]
    for threshold in dict.fromkeys([0.5, selected_threshold]):

        y_pred = [
            "spam" if prob >= threshold else "ham"
            for prob in spam_probs
        ]

        tn, fp, fn, tp = confusion_matrix(
            y_test,
            y_pred,
            labels=["ham", "spam"]
        ).ravel()

        final_results.append({
            "Model": model_name,
            "Threshold": threshold,
            "Selected on validation": threshold == selected_threshold,
            "Accuracy": accuracy_score(y_test, y_pred),
            "Precision": precision_score(
                y_test, y_pred,
                pos_label="spam", zero_division=0
            ),
            "Recall": recall_score(
                y_test, y_pred,
                pos_label="spam", zero_division=0
            ),
            "F1-score": f1_score(
                y_test, y_pred,
                pos_label="spam", zero_division=0
            ),
            "False Positive": fp,
            "False Negative": fn
        })

final_df = pd.DataFrame(final_results)

final_df.to_csv(OUTPUTS_DIR / "threshold_metrics.csv", index=False)
display(final_df.round(4))

,Model,Threshold,Selected on validation,Accuracy,Precision,Recall,F1-score,False Positive,False Negative
0,Multinomial Naive Bayes,0.5,False,0.9633,1.0000,0.7099,0.8304,0,38
1,Multinomial Naive Bayes,0.3,True,0.9710,0.9903,0.7786,0.8718,1,29
2,Logistic Regression,0.5,False,0.9633,1.0000,0.7099,0.8304,0,38
3,Logistic Regression,0.3,True,0.9768,0.9496,0.8626,0.9040,6,18


### 7.6. Đọc kết quả điều chỉnh ngưỡng

Cột **Selected on validation** đánh dấu ngưỡng có F1-score cao nhất trên Validation của từng mô hình.
Bảng bên trên so sánh ngưỡng mặc định 0.50 và ngưỡng đã chọn trên tập Test.
So sánh **False Negative** để biết số Spam bị bỏ sót và **False Positive** để biết số Ham bị báo nhầm.
Các kết quả phụ thuộc vào bộ dữ liệu và cách chia tập.

### 7.7. Phân tích ba lỗi cụ thể của Logistic Regression

Ba tin dưới đây đến từ tập Test đã chia. Chúng minh họa ba tình huống khi đổi ngưỡng. Mỗi mức đóng góp là giá trị TF-IDF nhân hệ số Logistic Regression. Giá trị dương đẩy điểm về Spam, giá trị âm đẩy điểm về Ham. Bảng hiển thị tối đa ba đặc trưng mạnh nhất ở mỗi hướng.

In [18]:
model = trained_models["Logistic Regression"]
vectorizer = model.named_steps["tfidf"]
classifier = model.named_steps["classifier"]
feature_names = vectorizer.get_feature_names_out()
example_ids = [4111, 1031, 3743]
messages = X_test.loc[example_ids]
vectors = vectorizer.transform(messages)
spam_idx = list(model.classes_).index("spam")
spam_probabilities = model.predict_proba(messages)[:, spam_idx]
spam_weights = classifier.coef_[0] if spam_idx == 1 else -classifier.coef_[0]

example_rows = []
for sample_id, message, vector, probability in zip(example_ids, messages, vectors, spam_probabilities):
    scores = sorted(
        (value * spam_weights[index], feature_names[index])
        for index, value in zip(vector.indices, vector.data)
    )
    toward_spam = [(term, round(score, 3)) for score, term in reversed(scores) if score > 0][:3]
    toward_ham = [(term, round(score, 3)) for score, term in scores if score < 0][:3]
    example_rows.append({
        "ID": sample_id,
        "SMS": message,
        "Nhãn thật": y_test.loc[sample_id],
        "Xác suất Spam": f"{probability:.2%}",
        "Nhãn ở 0.50": "spam" if probability >= 0.50 else "ham",
        "Nhãn ở 0.30": "spam" if probability >= 0.30 else "ham",
        "Nghiêng Spam": toward_spam,
        "Nghiêng Ham": toward_ham,
    })

display(pd.DataFrame(example_rows))


,ID,SMS,Nhãn thật,Xác suất Spam,Nhãn ở 0.50,Nhãn ở 0.30,Nghiêng Spam,Nghiêng Ham
0,4111,RECPT 1/3. You have ordered a Ringtone. Your o...,spam,31.82%,ham,spam,"[(you have, 0.491), (ringtone, 0.462), (your, ...","[(you, -0.084), (being, -0.027), (is being, -0..."
1,1031,I'm at work. Please call,ham,33.65%,ham,spam,"[(call, 1.254), (please call, 0.739), (please,...","[(at, -0.329), (work, -0.168), (at work, -0.027)]"
2,3743,ringtoneking 84484,spam,7.49%,ham,ham,[],[]


- **4111 — Spam bị bỏ sót ở 0,50.** Các đặc trưng `you have`, `ringtone` và `your` đẩy điểm về Spam. Cách viết giống thông báo xác nhận đơn hàng có thể khiến nội dung khó phân biệt. Xác suất **31,82%** đủ để phát hiện tin này ở ngưỡng 0,30.
- **1031 — Ham bị báo nhầm ở 0,30.** Trong câu hội thoại về việc gọi lại, `call` và `please call` đẩy điểm về Spam, còn `at` và `work` đẩy về Ham. Xác suất **33,65%** nằm giữa hai ngưỡng.
- **3743 — Spam bị bỏ sót ở cả hai ngưỡng.** Không có từ hay cụm từ nào của tin nằm trong từ vựng TF-IDF đã học. Vector toàn số 0, nên xác suất **7,49%** đến từ hệ số chặn của Logistic Regression. Hạ ngưỡng xuống 0,30 vẫn bỏ sót mẫu này.